<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/03_product_qna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Models

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add your API key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`. The code below will then retrieve it.

In [ ]:
#!pip uninstall -y langchain langchain-core langchain-community langgraph langchain-openai langchain-google-genai google-generativeai langchain-chroma
#!pip install -U tenacity
#!pip install -U langchain langchain-core langchain-community langgraph
#!pip install -U langchain-google-genai google-generativeai
#!pip install -U pysqlite3-binary langchain-chroma pandas pypdf nbformat

from google.colab import userdata
import os

api_key = userdata.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Missing GOOGLE_API_KEY in Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = api_key

print("Loaded Gemini key suffix:", api_key[-6:])

Loaded Gemini key suffix: PRESBs


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

03.02. Add Product Pricing function tool

In [ ]:
import pandas as pd
from langchain_core.tools import tool

#Load the laptop product pricing CSV into a Pandas dataframe.
import os
pricing_path = "Laptop pricing.csv"
if not os.path.exists(pricing_path):
    pricing_path = "/content/Laptop pricing.csv"
if not os.path.exists(pricing_path):
    pricing_path = "data/Laptop pricing.csv"
product_pricing_df = pd.read_csv(pricing_path)
print(product_pricing_df)

@tool
def get_laptop_price(laptop_name:str) -> int :
    """
    This function returns the price of a laptop, given its name as input.
    It performs a substring match between the input name and the laptop name.
    If a match is found, it returns the pricxe of the laptop.
    If there is NO match found, it returns -1
    """

    #Filter Dataframe for matching names
    match_records_df = product_pricing_df[
                        product_pricing_df["Name"].str.contains(
                                                "^" + laptop_name, case=False)
                        ]
    #Check if a record was found, if not return -1
    if len(match_records_df) == 0 :
        return -1
    else:
        return match_records_df["Price"].iloc[0]

#print(get_laptop_price("alpha"))
#print(get_laptop_price("testing"))


            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2


03.03. Add Product Features Retrieval Tool

In [ ]:
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

import os
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the product descriptions PDF
pdf_path = "Laptop product descriptions.pdf"
if not os.path.exists(pdf_path):
    pdf_path = "/content/Laptop product descriptions.pdf"
if not os.path.exists(pdf_path):
    pdf_path = "./data/Laptop product descriptions.pdf"

# Persist the vector store so 06 does not rebuild embeddings every time
persist_dir = "/content/chroma_product_store"

if os.path.exists(persist_dir) and len(os.listdir(persist_dir)) > 0:
    prod_feature_store = Chroma(
        persist_directory=persist_dir,
        embedding_function=embedding
    )
else:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,
        chunk_overlap=256
    )
    splits = text_splitter.split_documents(docs)

    prod_feature_store = Chroma.from_documents(
        documents=splits,
        embedding=embedding,
        persist_directory=persist_dir
    )

retriever = prod_feature_store.as_retriever(search_kwargs={"k": 1})

@tool
def get_product_features(query: str) -> str:
    """
    Look up laptop product details and features.
    Use this for laptop names, specs, memory, storage, design, and advantages.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No matching laptop product details were found."
    return "\n\n".join(doc.page_content for doc in docs)
